# exp_010 — train CNN+PPO / JEPA baselines on real LS20 (Google Colab)

Self-contained notebook to train the three exp_010 sub-experiments on a Colab **GPU** and download the resulting checkpoints so you can drop them straight back into the repo.

**What it trains**
- `exp_010_0_cnn_ppo_baseline` — vanilla CNN+PPO (terminal-only reward)
- `exp_010_1_jepa_joint_online` — joint online JEPA+PPO (on-policy encoder data)
- `exp_010_2_jepa_random_pretrain` — random-data JEPA pretrain → unfrozen PPO

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is fine).

**Two ways to get the code** (run *one* of Step 2A / 2B):
- **2A git clone** — easiest if you've pushed exp_010 to GitHub (`git push origin main`).
- **2B zip upload** — works with *uncommitted* local code; upload a zip you make locally.

**Artifacts:** Step 6 zips every checkpoint / metrics file with its **repo-relative path** preserved, so you unzip at your `Code Repo/` root and everything lands where the dashboard expects it.

## Step 1 — Install dependencies
torch + numpy are preinstalled on Colab; we only add the ARC-AGI engine. (Needs a recent Python; Colab's default is fine.)

In [ ]:
import sys, platform
print('Python', platform.python_version())
!pip -q install 'arc-agi>=0.9.8' 'arcengine>=0.9.3'
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > GPU'

## Step 2A — Get the code via git clone *(recommended)*
Push exp_010 to GitHub first. If the repo is **private**, paste a GitHub token when prompted (it is not stored); if **public**, just press Enter.

In [ ]:
import os, getpass
REPO_URL = 'github.com/LavetteSinsora/ProjectArceus.git'
BRANCH   = 'main'
token = getpass.getpass('GitHub token (press Enter if the repo is public): ').strip()
auth = f'{token}@' if token else ''
os.chdir('/content')
!rm -rf /content/ProjectArceus
!git clone --depth 1 --branch {BRANCH} https://{auth}{REPO_URL} /content/ProjectArceus
REPO_ROOT = '/content/ProjectArceus'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('repo at', REPO_ROOT)
assert os.path.exists('environment_files/ls20'), 'environment_files/ls20 missing — wrong repo/branch?'
assert os.path.exists('JEPA/experiments/exp_010_ls20_cnn_ppo_jepa'), 'exp_010 not in clone — did you push it?'

## Step 2B — *Alternative:* get the code via zip upload
Use this instead of 2A if you have **not** pushed exp_010. On your machine, from inside `Code Repo/`, make a zip with the two things Colab needs (the JEPA package + the LS20 env files):
```bash
zip -r exp010_code.zip JEPA environment_files pyproject.toml
```
Then run the cell below and upload `exp010_code.zip`.

In [ ]:
# --- Only run this cell if you skipped Step 2A ---
import os, sys, zipfile
from google.colab import files
os.makedirs('/content/ProjectArceus', exist_ok=True)
up = files.upload()  # choose exp010_code.zip
name = next(iter(up))
with zipfile.ZipFile(name) as z:
    z.extractall('/content/ProjectArceus')
REPO_ROOT = '/content/ProjectArceus'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
assert os.path.exists('environment_files/ls20') and os.path.exists('JEPA/experiments/exp_010_ls20_cnn_ppo_jepa')
print('repo ready at', REPO_ROOT)

## Step 3 — Verify the LS20 environment loads and steps

In [ ]:
import numpy as np
from JEPA.experiments.exp_010_ls20_cnn_ppo_jepa.shared.device import get_device
from JEPA.experiments.exp_010_ls20_cnn_ppo_jepa.shared.ls20_vec_env import VecLS20Env
print('device:', get_device())
_env = VecLS20Env('ls20', n_envs=2, max_episode_steps=64, seed=0)
obs, r, d, info = _env.step(np.zeros(2, dtype=np.int64))
print('obs', obs.shape, obs.dtype, '| n_actions', _env.n_actions)
del _env

## Step 4 — Configure the run
Defaults are sized for a single Colab session. Scale `TOTAL_ENV_STEPS` (and `N_RANDOM_TRANSITIONS`) up for real results — the local SYSTEM_CARD defaults are 3M env steps / 500k transitions.

Set the `RUN_*` flags to choose which sub-experiments to train.

In [ ]:
RUN_010_0 = True   # CNN+PPO baseline
RUN_010_1 = True   # joint online JEPA+PPO
RUN_010_2 = True   # random-data JEPA pretrain -> unfrozen PPO

TOTAL_ENV_STEPS       = 500_000   # per PPO run (bump up for real results)
N_RANDOM_TRANSITIONS  = 200_000   # exp_010_2 random buffer size
N_ENVS                = 8
EVAL_EVERY            = 50        # updates
SAVE_EVERY            = 100       # updates
import dataclasses
def with_overrides(cfg):
    return dataclasses.replace(cfg, total_env_steps=TOTAL_ENV_STEPS, n_envs=N_ENVS,
                               eval_every=EVAL_EVERY, save_every=SAVE_EVERY)

## Step 5 — Train

In [ ]:
# exp_010_0 — CNN+PPO baseline
if RUN_010_0:
    from JEPA.experiments.exp_010_ls20_cnn_ppo_jepa.shared.trainer import train
    from JEPA.experiments.exp_010_ls20_cnn_ppo_jepa.exp_010_0_cnn_ppo_baseline.config import Config as C0
    train(with_overrides(C0()))

In [ ]:
# exp_010_1 — joint online JEPA + PPO
if RUN_010_1:
    from JEPA.experiments.exp_010_ls20_cnn_ppo_jepa.shared.trainer import train
    from JEPA.experiments.exp_010_ls20_cnn_ppo_jepa.exp_010_1_jepa_joint_online.config import Config as C1
    train(with_overrides(C1()))

In [ ]:
# exp_010_2 — collect random data -> pretrain JEPA -> unfrozen PPO
if RUN_010_2:
    import dataclasses
    from JEPA.experiments.exp_010_ls20_cnn_ppo_jepa.shared.pretrain import collect_random, pretrain_jepa, _repo_root
    from JEPA.experiments.exp_010_ls20_cnn_ppo_jepa.shared.trainer import train
    from JEPA.experiments.exp_010_ls20_cnn_ppo_jepa.exp_010_2_jepa_random_pretrain.config import Config as C2
    from JEPA.experiments.exp_010_ls20_cnn_ppo_jepa.exp_010_2_jepa_random_pretrain.collect import buffer_path
    cfg2 = dataclasses.replace(with_overrides(C2()), n_random_transitions=N_RANDOM_TRANSITIONS)
    bp = buffer_path(cfg2)
    print('--- collecting random data ---')
    collect_random(cfg2, cfg2.n_random_transitions, bp)
    print('--- pretraining JEPA ---')
    pretrain_jepa(cfg2, bp)
    enc = _repo_root() / cfg2.exp_dir / 'jepa_pretrained' / 'encoder_final.pt'
    print('--- PPO from pretrained encoder (unfrozen) ---')
    train(dataclasses.replace(cfg2, init_encoder_ckpt=str(enc), freeze_encoder=False))

## Step 6 — Package & download artifacts
Zips checkpoints, metrics, the pretrained encoder, and the random-buffer meta — each with its **repo-relative path**. Unzip at your local `Code Repo/` root to place everything where the dashboard reads it.

In [ ]:
import os, glob, zipfile
EXP = 'JEPA/experiments/exp_010_ls20_cnn_ppo_jepa'
patterns = [
    f'{EXP}/*/checkpoints/*.pt',
    f'{EXP}/*/runs/*/metrics.jsonl',
    f'{EXP}/*/runs/*/config.json',
    f'{EXP}/*/jepa_pretrained/*.pt',
    f'{EXP}/*/data/*.meta.json',
]
files_to_zip = sorted({p for pat in patterns for p in glob.glob(os.path.join(REPO_ROOT, pat))})
out_zip = '/content/exp_010_artifacts.zip'
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in files_to_zip:
        z.write(p, arcname=os.path.relpath(p, REPO_ROOT))  # repo-relative path
print(f'{len(files_to_zip)} files -> {out_zip}')
for p in files_to_zip:
    print('  ', os.path.relpath(p, REPO_ROOT))
from google.colab import files
files.download(out_zip)

### Placing the artifacts back locally
From your local `Code Repo/` root:
```bash
unzip -o ~/Downloads/exp_010_artifacts.zip
```
Checkpoints land in `JEPA/experiments/exp_010_ls20_cnn_ppo_jepa/<sub_exp>/checkpoints/` and metrics in `.../runs/<run>/`. Then launch the dashboard and inspect:
```bash
uv run python JEPA/dashboard/server.py   # http://localhost:8787
```
Pick `exp_010_ls20_cnn_ppo_jepa/<sub_exp>` to play a checkpoint or view training curves.